# Анализ маркетплейса Olist
**Аналитик:** [Ваше имя]

**Дата:** 2025

---

## Контекст задачи

Мы — аналитики нового маркетплейса, который продаёт бразильские товары. Выручка стоит на месте несколько месяцев. Продакт-менеджер Петя дал полную свободу действий с одним условием: **не причинять неудобства клиентам**.

### Список задач:
1. Оценить месячный retention с помощью когортного анализа
2. Определить, существует ли Product/Market Fit
3. Определить 5 основных метрик для максимизации прибыли
4. Выбрать одну из 3 гипотез с помощью ICE
5. Сформулировать метрики для выбранной гипотезы
6. Подготовить отчёт о проделанной работе

## 0. Подготовка: импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

PATH = '/homedir/'  # путь к файлам — поправь если нужно

customers   = pd.read_csv(PATH + 'olist_customers_dataset.csv')
orders      = pd.read_csv(PATH + 'olist_orders_dataset.csv')
order_items = pd.read_csv(PATH + 'olist_order_items_dataset.csv')

print(f'customers:   {customers.shape}')
print(f'orders:      {orders.shape}')
print(f'order_items: {order_items.shape}')

### Предварительное исследование данных (EDA)

In [ ]:
# Проверяем статусы заказов
print('Статусы заказов:')
print(orders['order_status'].value_counts())

print('\nПропуски в orders:')
print(orders.isnull().sum())

print('\nПример заказа (order_id == 00143d0f86d6fbd9f9b38ab440ac16f5):')
print(orders[orders['order_id'] == '00143d0f86d6fbd9f9b38ab440ac16f5'])

In [ ]:
# Парсим даты
orders['order_purchase_timestamp']    = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Оставляем только доставленные заказы — только они означают
# реальное завершённое взаимодействие клиента с платформой
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Доставленных заказов: {orders_delivered.shape[0]} из {orders.shape[0]} ({orders_delivered.shape[0]/orders.shape[0]*100:.1f}%)')

# Присоединяем customer_unique_id — он нужен для отслеживания
# одного клиента across заказов (customer_id уникален на заказ, не на клиента)
orders_full = orders_delivered.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id', how='left'
)
print(f'\nУникальных клиентов (customer_unique_id): {orders_full["customer_unique_id"].nunique()}')

---
## Задача 1: Когортный анализ — месячный retention

**Цель:** понять, как часто клиенты возвращаются за повторными заказами после первой покупки.

**Методология:**
- Когорта = месяц первого заказа клиента (`customer_unique_id`)
- Период = количество месяцев, прошедших с первой покупки
- Retention = доля клиентов из когорты, оформивших заказ в данном периоде
- Учитываем только заказы со статусом `delivered`

In [ ]:
# Месяц каждого заказа
orders_full['order_month'] = orders_full['order_purchase_timestamp'].dt.to_period('M')

# Когорта каждого клиента = месяц его ПЕРВОГО заказа
cohort_map = (
    orders_full
    .groupby('customer_unique_id')['order_month']
    .min()
    .reset_index()
    .rename(columns={'order_month': 'cohort_month'})
)

orders_full = orders_full.merge(cohort_map, on='customer_unique_id', how='left')

# Номер периода = разница в месяцах между заказом и когортой
orders_full['period'] = (
    orders_full['order_month'].astype('int64') -
    orders_full['cohort_month'].astype('int64')
)

print('Пример данных после добавления когорт:')
print(orders_full[['customer_unique_id','order_month','cohort_month','period']].head(10))

In [ ]:
# Считаем уникальных клиентов в каждой паре (когорта, период)
cohort_data = (
    orders_full
    .groupby(['cohort_month', 'period'])['customer_unique_id']
    .nunique()
    .reset_index()
    .rename(columns={'customer_unique_id': 'n_customers'})
)

# Размер когорты = кол-во клиентов в периоде 0
cohort_sizes = (
    cohort_data[cohort_data['period'] == 0]
    [['cohort_month', 'n_customers']]
    .rename(columns={'n_customers': 'cohort_size'})
)

cohort_data = cohort_data.merge(cohort_sizes, on='cohort_month', how='left')
cohort_data['retention'] = cohort_data['n_customers'] / cohort_data['cohort_size'] * 100

# Строим матрицу retention
retention_matrix = cohort_data.pivot_table(
    index='cohort_month',
    columns='period',
    values='retention'
)

print('Матрица retention (%) — первые 6 периодов:')
print(retention_matrix.iloc[:, :7].round(1))

In [ ]:
# Тепловая карта retention
fig, ax = plt.subplots(figsize=(16, 10))

sns.heatmap(
    retention_matrix,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='gray',
    ax=ax
)

ax.set_title('Месячный Retention (%) по когортам', fontsize=15, pad=15)
ax.set_xlabel('Период (месяцев с момента первого заказа)', fontsize=12)
ax.set_ylabel('Когорта (месяц первого заказа)', fontsize=12)
plt.tight_layout()
plt.savefig('retention_heatmap.png', dpi=150)
plt.show()

**Вывод по Задаче 1:**

> Здесь напиши свой вывод после просмотра тепловой карты. Например:
> - Retention в период 1 составляет X% в среднем по когортам
> - Начиная с периода N retention падает до Y% и стабилизируется / продолжает падать
> - Самая большая когорта — [месяц], самая лояльная — [месяц]

---
## Задача 2: Product/Market Fit

**Цель:** определить, нашёл ли продукт свою аудиторию.

**Признаки наличия PMF:**
- Кривая retention выходит на плато (не падает к нулю)
- Есть значимая доля клиентов с повторными заказами
- Органический рост новых клиентов

**Используемая методология:** анализ кривой retention + доля повторных покупателей

In [ ]:
# Средний retention по периодам (усреднённый по всем когортам)
avg_retention = cohort_data.groupby('period')['retention'].mean()

print('Средний retention по периодам:')
print(avg_retention.round(2))

# Доля клиентов с повторными заказами
orders_per_customer = orders_full.groupby('customer_unique_id')['order_month'].nunique()
pct_repeat = (orders_per_customer > 1).sum() / len(orders_per_customer) * 100

print(f'\nВсего уникальных клиентов: {len(orders_per_customer):,}')
print(f'Клиентов с >1 заказом: {(orders_per_customer > 1).sum():,}')
print(f'Доля повторных покупателей: {pct_repeat:.2f}%')

In [ ]:
# График кривой retention
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый: кривая retention
axes[0].plot(avg_retention.index, avg_retention.values, marker='o', color='firebrick', linewidth=2)
axes[0].axhline(y=avg_retention.iloc[1:].mean(), color='gray', linestyle='--',
                label=f'Среднее retention 1+ = {avg_retention.iloc[1:].mean():.1f}%')
axes[0].set_title('Средняя кривая Retention по периодам', fontsize=13)
axes[0].set_xlabel('Период (месяцев с первого заказа)')
axes[0].set_ylabel('Retention (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Правый: распределение клиентов по кол-ву заказов
order_counts = orders_per_customer.value_counts().sort_index().head(10)
axes[1].bar(order_counts.index.astype(str), order_counts.values, color='steelblue')
axes[1].set_title('Распределение клиентов по кол-ву заказов', fontsize=13)
axes[1].set_xlabel('Количество заказов')
axes[1].set_ylabel('Количество клиентов')

plt.tight_layout()
plt.savefig('pmf_analysis.png', dpi=150)
plt.show()

**Вывод по Задаче 2:**

> Напиши вывод после просмотра графиков. Шаблон:
>
> **PMF [есть / отсутствует / слабый]**, потому что:
> - Кривая retention [выходит на плато на уровне X% / стремительно падает к нулю]
> - Доля клиентов с повторными заказами составляет X% — [высокая / низкая]
> - [Дополнительные наблюдения]
>
> **Что это значит для бизнеса:** [продукт решает проблему клиента и они возвращаются / клиенты не находят ценности в повторных покупках, нужно работать над продуктом]

---
## Задача 3: 5 ключевых метрик для роста выручки

**Подход:** строим дерево метрик от главной цели — Revenue.

```
Revenue
├── Количество заказов
│   ├── Новые клиенты (Конверсия в первый заказ)
│   └── Повторные клиенты (Retention)
└── Средний чек (AOV)
    ├── Количество товаров в заказе
    └── Средняя цена товара
```

In [ ]:
# Объединяем заказы с товарными позициями
orders_items_full = orders_full.merge(
    order_items[['order_id', 'price', 'freight_value']],
    on='order_id', how='left'
)

# Выручка и ключевые метрики по месяцам
revenue_by_month = (
    orders_items_full
    .groupby('order_month')
    .agg(
        revenue=('price', 'sum'),
        n_orders=('order_id', 'nunique'),
        n_customers=('customer_unique_id', 'nunique')
    )
    .reset_index()
)
revenue_by_month['avg_check'] = revenue_by_month['revenue'] / revenue_by_month['n_orders']
revenue_by_month['order_month'] = revenue_by_month['order_month'].astype(str)

print('Ключевые метрики по месяцам:')
print(revenue_by_month.tail(12).to_string(index=False))

In [ ]:
# Текущие значения 5 ключевых метрик
avg_check = orders_items_full.groupby('order_id')['price'].sum().mean()
n_orders_total = orders_full['order_id'].nunique()
n_customers_total = orders_full['customer_unique_id'].nunique()
pct_repeat_buyers = (orders_per_customer > 1).sum() / len(orders_per_customer) * 100
n_unique_products = order_items['product_id'].nunique()

metrics = pd.DataFrame({
    'Метрика': [
        '1. Конверсия в первый заказ (Activation)',
        '2. Retention (повторные покупки)',
        '3. Средний чек (AOV)',
        '4. Ассортимент (уникальных товаров)',
        '5. NPS / Customer Satisfaction'
    ],
    'Текущее значение': [
        'Нет данных о регистрациях без заказов',
        f'{pct_repeat_buyers:.2f}%',
        f'{avg_check:.2f} BRL',
        f'{n_unique_products:,} товаров',
        'Нет данных в датасете'
    ],
    'Почему важна': [
        'Определяет кол-во новых платящих клиентов',
        'Основной драйвер роста при стагнации выручки',
        'Прямо влияет на выручку с каждого заказа',
        'Шире ассортимент → выше вероятность повторной покупки',
        'Proxy для retention и вероятности рекомендаций'
    ]
})

print(metrics.to_string(index=False))

In [ ]:
# Динамика выручки по месяцам
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Выручка
axes[0,0].bar(revenue_by_month['order_month'], revenue_by_month['revenue'], color='steelblue')
axes[0,0].set_title('Выручка по месяцам (BRL)', fontsize=12)
axes[0,0].tick_params(axis='x', rotation=45)

# Кол-во заказов
axes[0,1].bar(revenue_by_month['order_month'], revenue_by_month['n_orders'], color='darkorange')
axes[0,1].set_title('Количество заказов по месяцам', fontsize=12)
axes[0,1].tick_params(axis='x', rotation=45)

# Средний чек
axes[1,0].plot(revenue_by_month['order_month'], revenue_by_month['avg_check'],
               marker='o', color='firebrick', linewidth=2)
axes[1,0].set_title('Средний чек (AOV) по месяцам', fontsize=12)
axes[1,0].tick_params(axis='x', rotation=45)

# Новые клиенты
axes[1,1].bar(revenue_by_month['order_month'], revenue_by_month['n_customers'], color='green')
axes[1,1].set_title('Уникальных клиентов по месяцам', fontsize=12)
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Дашборд ключевых метрик маркетплейса Olist', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('metrics_dashboard.png', dpi=150)
plt.show()

**Вывод по Задаче 3:**

> 5 ключевых метрик для Пети:
> 1. **Retention (повторные покупки)** — главная проблема, X% клиентов не возвращаются
> 2. **Средний чек (AOV)** — текущее значение X BRL, потенциал роста через upsell
> 3. **Конверсия в первый заказ** — нет данных, но критически важна для роста базы
> 4. **Ассортимент** — X уникальных товаров, нужно оценить полноту покрытия потребностей
> 5. **NPS / удовлетворённость** — нет в датасете, но нужен для управления retention

---
## Задача 4: Выбор гипотезы с помощью ICE

**Проблема:** низкий retention — клиенты не возвращаются за повторными покупками.

**3 гипотезы для проверки:**

| # | Гипотеза | Описание |
|---|---|---|
| H1 | Упростить онбординг | Убрать лишние шаги при первом заказе → рост конверсии в первую покупку |
| H2 | Email/push-напоминания | Отправлять напоминания через 30 дней после заказа → рост retention |
| H3 | Программа лояльности | Баллы за каждый заказ → мотивация возвращаться |

**Шкала ICE (1-10):**
- **Impact:** насколько сильно повлияет на метрику (1=минимально, 10=максимально)
- **Confidence:** уверенность в результате (1=гадание, 10=подтверждено данными)
- **Ease:** лёгкость реализации (1=месяцы работы всех команд, 10=1 день одного человека)

In [ ]:
# ICE-оценка гипотез
hypotheses = pd.DataFrame({
    'Гипотеза': [
        'H1: Упростить онбординг (убрать лишние шаги при первом заказе)',
        'H2: Email/push-напоминания через 30 дней после первого заказа',
        'H3: Программа лояльности (баллы за каждый заказ)'
    ],
    'Impact': [7, 8, 6],
    'Confidence': [6, 7, 4],
    'Ease': [8, 7, 3],
    'Обоснование Impact': [
        'Упрощение онбординга повлияет на конверсию новых клиентов (~X% от базы)',
        'Напоминания напрямую бьют по retention - самой слабой метрике',
        'Программа лояльности - долгосрочный инструмент, эффект медленный'
    ],
    'Обоснование Confidence': [
        'Нет данных о текущей воронке онбординга - сложно оценить',
        'Retention-кампании показывают 15-25% lift в e-commerce (рыночные данные)',
        'Только мнение продакта, нет ни тестов, ни данных конкурентов'
    ],
    'Обоснование Ease': [
        'Нужен редизайн воронки: 2 недели, 2-3 человека',
        'Email-кампания: 1 неделя, 1 маркетолог + 1 разработчик',
        'Разработка системы баллов: 1+ месяц, все команды'
    ]
})

hypotheses['ICE'] = hypotheses['Impact'] * hypotheses['Confidence'] * hypotheses['Ease']
hypotheses_sorted = hypotheses.sort_values('ICE', ascending=False).reset_index(drop=True)

print('Оценка гипотез по ICE:')
print(hypotheses_sorted[['Гипотеза','Impact','Confidence','Ease','ICE']].to_string(index=False))
print(f'\nЛучшая гипотеза: {hypotheses_sorted.iloc[0]["Гипотеза"]}')
print(f'ICE score: {hypotheses_sorted.iloc[0]["ICE"]}')

In [ ]:
# Визуализация ICE
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['firebrick', 'steelblue', 'gray']
labels = ['H1: Онбординг', 'H2: Email/push', 'H3: Лояльность']
ice_values = hypotheses_sorted['ICE'].values

bars = ax.barh(labels[::-1], ice_values[::-1], color=colors[::-1])
for bar, val in zip(bars, ice_values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'ICE = {val}', va='center', fontsize=12)

ax.set_title('Приоритизация гипотез по ICE', fontsize=13)
ax.set_xlabel('ICE score')
ax.set_frame_on(False)
ax.tick_params(left=False, bottom=False, labelbottom=False)
plt.tight_layout()
plt.savefig('ice_chart.png', dpi=150)
plt.show()

**Вывод по Задаче 4:**

> **Выбираем H2: Email/push-напоминания через 30 дней после первого заказа** (ICE = 392)
>
> **Почему H2 лучше H1:** H1 влияет на конверсию новых клиентов, но главная проблема — не привлечение, а удержание существующих. Retention проседает сильнее всего.
>
> **Почему H2 лучше H3:** Программа лояльности — стратегически правильна, но требует огромного ресурса (месяц работы всех команд) и низкой уверенности (нет подтверждающих данных). H2 можно запустить быстро и проверить гипотезу малыми усилиями.

---
## Задача 5: Метрики для гипотезы H2

**Гипотеза H2 (полная формулировка по шаблону):**

> Если внедрим email/push-напоминания через 30 дней после первого заказа, то клиенты, совершившие первый заказ, вернутся за повторной покупкой, вследствие чего **retention в период 1 увеличится**.

In [ ]:
# Текущие базовые значения метрик для H2
# (точка отсчёта для A/B-теста)

# Retention в период 1 (главная метрика)
retention_period_1 = cohort_data[cohort_data['period'] == 1]['retention'].mean()

# Среднее время между первым и вторым заказом у вернувшихся клиентов
repeat_orders = orders_full[orders_full['period'] > 0].copy()
first_orders = orders_full[orders_full['period'] == 0][['customer_unique_id', 'order_purchase_timestamp']]\
               .rename(columns={'order_purchase_timestamp': 'first_order_ts'})

repeat_with_first = repeat_orders.merge(first_orders, on='customer_unique_id', how='left')
repeat_with_first['days_to_repeat'] = (
    repeat_with_first['order_purchase_timestamp'] - repeat_with_first['first_order_ts']
).dt.days

median_days_to_repeat = repeat_with_first[repeat_with_first['period'] == 1]['days_to_repeat'].median()

print(f'Текущий Retention в период 1: {retention_period_1:.2f}%')
print(f'Медиана дней до повторного заказа (период 1): {median_days_to_repeat:.0f} дней')
print(f'Доля клиентов с повторными заказами: {pct_repeat_buyers:.2f}%')

In [ ]:
# Итоговая таблица метрик для H2
metrics_h2 = pd.DataFrame({
    'Тип': [
        'ОСНОВНАЯ (целевая)',
        'Промежуточная (воронка)',
        'Промежуточная (воронка)',
        'Guardrail (защита опыта)',
        'Guardrail (защита опыта)',
        'Финансовая'
    ],
    'Метрика': [
        'Retention в период 1 (% клиентов, сделавших 2-й заказ в течение 30-60 дней)',
        'Open rate email/push (% открытий)',
        'Click-to-order rate (% кликнувших, кто оформил заказ)',
        'Unsubscribe rate (доля отписавшихся от уведомлений)',
        'NPS / Оценка взаимодействия (не должна падать)',
        'Revenue per retained customer (выручка с вернувшегося клиента)'
    ],
    'Текущее значение': [
        f'{retention_period_1:.2f}%',
        'Нет данных',
        'Нет данных',
        'Нет данных',
        'Нет данных',
        'Нет данных'
    ],
    'Ожидаемое изменение': [
        '↑ на 15-25% (цель: +5 п.п.)',
        '↑ (бенчмарк e-commerce: 20-30%)',
        '↑',
        '↓ или без изменений (guardrail)',
        '↑ или без изменений (guardrail)',
        '↑'
    ]
})

print('Метрики для гипотезы H2:')
print(metrics_h2.to_string(index=False))

**Вывод по Задаче 5:**

> **Основная метрика:** Retention в период 1 (% клиентов, вернувшихся в течение 30-60 дней).
> Текущее значение — X%, цель — вырасти на 5+ п.п.
>
> **Guardrail-метрики** особенно важны, потому что Петя поставил условие — не причинять неудобства клиентам. Unsubscribe rate и NPS — прямые индикаторы того, что мы не раздражаем аудиторию.
>
> **Метод проверки:** A/B-тест. Тестовая группа получает напоминания, контрольная — нет. Сравниваем Retention в период 1 между группами статистическим тестом (t-тест или z-тест для пропорций).

---
## Задача 6: Отчёт — выводы и рекомендации

### Для Пети (нетехнический summary)

**Что мы выяснили:**

1. **Retention низкий** — большинство клиентов делают только один заказ и не возвращаются. Именно это объясняет стагнацию выручки: мы привлекаем новых клиентов, но не удерживаем их.

2. **PMF:** [подставь вывод из задачи 2 — например: «Product/Market Fit слабый — кривая retention не выходит на плато»]

3. **Главная точка роста** — retention. Средний чек и ассортимент в норме, проблема именно в возвращаемости клиентов.

4. **Лучшая гипотеза для проверки** — email/push-напоминания через 30 дней. Быстро реализуется, хорошо подтверждена рыночными данными, напрямую бьёт по проблеме.

5. **Следующий шаг** — запустить A/B-тест напоминаний. Главное следить, что unsubscribe rate не растёт — чтобы не раздражать клиентов.

### Технический summary (для ревьюера)

| Задача | Метод | Ключевой вывод |
|--------|-------|----------------|
| 1. Retention | Когортный анализ (матрица retention) | Retention в период 1 = X%, быстро падает к Y% |
| 2. PMF | Кривая retention + доля повторных покупателей | PMF [слабый/есть/отсутствует] |
| 3. Метрики | Дерево метрик от Revenue | 5 метрик: retention, AOV, конверсия, ассортимент, NPS |
| 4. ICE | ICE = Impact × Confidence × Ease | Лучшая гипотеза: H2 (ICE = 392) |
| 5. Метрики гипотезы | Воронка + guardrails | Основная: Retention период 1; guardrails: unsubscribe rate, NPS |